# BingePlay Streaming Analytics — Advanced SQL Minor Project
The Unlox Academy · Data Analytics & Data Science · Week 4

**Submitted by: Sanjay VA**

Connects to the `bingeplay` MySQL/MariaDB database via SQLAlchemy + PyMySQL. One markdown answer cell and one SQL cell per question. Set `BINGEPLAY_URL` to point at your own server.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
engine = create_engine(os.environ.get("BINGEPLAY_URL", "mysql+pymysql://root:@127.0.0.1:3306/bingeplay"))

pd.read_sql("""
SELECT 'users' AS tbl, COUNT(*) AS n FROM users
UNION ALL SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL SELECT 'shows', COUNT(*) FROM shows
UNION ALL SELECT 'watch_sessions', COUNT(*) FROM watch_sessions
UNION ALL SELECT 'ratings', COUNT(*) FROM ratings
UNION ALL SELECT 'null_user_sessions', COUNT(*) FROM watch_sessions WHERE user_id IS NULL
""", engine)

,tbl,n
0,users,3000
1,subscriptions,4497
2,shows,100
3,watch_sessions,100351
4,ratings,5000
5,null_user_sessions,2


## Q1 — Active revenue
**Answer: 2,340 active subscriptions · ₹784,260 MRR**

Counts `status='active'` rows with `end_date IS NULL OR end_date > '2024-06-30'`. The NULL check matters: every open-ended subscription has a NULL end_date.

In [2]:
query = """
-- Q1: Active subscriptions & MRR as of 30-Jun-2024
SELECT COUNT(*)                AS active_subscriptions,
       SUM(monthly_price_inr)  AS total_mrr_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""
pd.read_sql(query, engine)

,active_subscriptions,total_mrr_inr
0,2340,784260.0


## Q2 — Signup momentum
**Answer: May & June — 600 signups each (highest)**

Signups climb every month from 350 in January to 600 in June.

In [3]:
query = """
-- Q2: Signups per month (Jan-Jun 2024)
SELECT MONTH(signup_date)        AS month,
       MONTHNAME(signup_date)    AS month_name,
       COUNT(*)                  AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY month;
"""
pd.read_sql(query, engine)

,month,month_name,signup_count
0,1,January,350
1,2,February,400
2,3,March,500
3,4,April,550
4,5,May,600
5,6,June,600


## Q3 — Device analytics
**Answer: see table below**

Sessions with NULL `user_id` (2 rows) are excluded. Mobile dominates volume, but average session length (~30 min) and completion rate (~60%) are nearly identical across devices — device doesn't change engagement quality.

In [4]:
query = """
-- Q3: Device analytics (NULL user_id sessions excluded)
SELECT device_type,
       COUNT(*)                                   AS total_sessions,
       SUM(watch_minutes)                         AS total_watch_minutes,
       ROUND(AVG(watch_minutes), 2)               AS avg_minutes_per_session,
       ROUND(100 * AVG(completed = 1), 2)         AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""
pd.read_sql(query, engine)

,device_type,total_sessions,total_watch_minutes,avg_minutes_per_session,completion_rate_pct
0,Mobile,50172,1504355.0,29.98,60.24
1,TV,27981,840595.0,30.04,59.98
2,Laptop,15105,453434.0,30.02,60.51
3,Tablet,7091,210733.0,29.72,59.79


## Q4 — Rating distribution
**Answer: 71.34% of ratings are 4 or 5 stars**

Distribution is heavily skewed positive; 1–2 star ratings together are only 11.72%.

In [5]:
query = """
-- Q4: Rating distribution
SELECT stars,
       COUNT(*) AS rating_count,
       ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS pct_of_ratings
FROM ratings
GROUP BY stars
ORDER BY stars;
"""
pd.read_sql(query, engine)

,stars,rating_count,pct_of_ratings
0,1,234,4.68
1,2,352,7.04
2,3,847,16.94
3,4,1781,35.62
4,5,1786,35.72


In [6]:
query_4b = """
-- Q4b: share of ratings that are 4 or 5 stars
SELECT ROUND(100 * AVG(stars >= 4), 2) AS pct_4_or_5_stars
FROM ratings;
"""
pd.read_sql(query_4b, engine)

,pct_4_or_5_stars
0,71.34


## Q5 — Originals vs acquired
**Answer: Originals perform better — avg IMDb 7.92 vs 6.63 (+1.29)**

30 originals vs 70 acquired shows; average release year is almost the same (~2020), so the gap isn't explained by recency.

In [7]:
query = """
-- Q5: Originals vs acquired
SELECT CASE is_original WHEN 1 THEN 'Original' ELSE 'Acquired' END AS content_type,
       COUNT(*)                        AS num_shows,
       ROUND(AVG(imdb_rating), 2)      AS avg_imdb_rating,
       ROUND(AVG(release_year), 2)     AS avg_release_year
FROM shows
GROUP BY is_original;
"""
pd.read_sql(query, engine)

,content_type,num_shows,avg_imdb_rating,avg_release_year
0,Acquired,70,6.63,2020.73
1,Original,30,7.92,2020.37


## Q6 — Binge day detection
**Answer: 414 binge days in Q2 2024**

**Top user: U02956 with 8 binge days.**

Grouped by (user, show, date) — grouping by user+date alone would wrongly count multi-show days.

In [8]:
query = """
-- Q6: Binge days in Q2 2024 (a binge day = user + show + date with 5+ sessions)
WITH binge_days AS (
    SELECT user_id, show_id, session_date, COUNT(*) AS sessions_that_day
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
per_user AS (
    SELECT user_id, COUNT(*) AS binge_day_count
    FROM binge_days
    GROUP BY user_id
)
SELECT (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
       user_id                           AS top_user,
       binge_day_count                   AS top_user_binge_days
FROM per_user
ORDER BY binge_day_count DESC, user_id
LIMIT 5;
"""
pd.read_sql(query, engine)

,total_binge_days,top_user,top_user_binge_days
0,414,U02956,8
1,414,U02292,6
2,414,U00118,5
3,414,U02334,5
4,414,U02169,4


## Q7 — Q1 signups who never watched (NULL trap)
**Answer: 226 users**

**Total Q1 signups: 1250** (18.1% never watched).

Uses `NOT EXISTS`. `NOT IN (SELECT user_id ...)` would return 0 rows because `watch_sessions.user_id` contains NULLs.

In [9]:
query = """
-- Q7: Q1-2024 signups who never watched (NOT EXISTS avoids the NOT IN + NULL trap)
SELECT COUNT(*) AS total_q1_signups,
       SUM(NOT EXISTS (SELECT 1 FROM watch_sessions ws
                       WHERE ws.user_id = u.user_id)) AS never_watched
FROM users u
WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-04-01';
"""
pd.read_sql(query, engine)

,total_q1_signups,never_watched
0,1250,226.0


## Q8 — Over-paying Premium/Family users
**Answer: 6 users**

Premium/Family users (current plan = latest subscription open on 30-Jun-2024) who have watched at least one show and **none** with `min_plan` of Premium/Family. Prime downgrade-offer targets. Users with zero sessions are excluded (nothing to say they only watch Basic content).

In [10]:
query = """
-- Q8: Premium/Family users (as of 30-Jun-2024) who only ever watched Basic-tier shows
WITH current_sub AS (            -- most recent subscription open on 30-Jun-2024
    SELECT s.*,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date DESC, subscription_id DESC) AS rn
    FROM subscriptions s
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
),
upper_plan_users AS (
    SELECT user_id FROM current_sub
    WHERE rn = 1 AND plan IN ('Premium', 'Family')
)
SELECT COUNT(*) AS overpaying_users
FROM upper_plan_users p
WHERE EXISTS (SELECT 1 FROM watch_sessions ws WHERE ws.user_id = p.user_id)   -- has watched something
  AND NOT EXISTS (SELECT 1
                  FROM watch_sessions ws
                  JOIN shows sh ON sh.show_id = ws.show_id
                  WHERE ws.user_id = p.user_id
                    AND sh.min_plan IN ('Premium', 'Family'));
"""
pd.read_sql(query, engine)

,overpaying_users
0,6


## Q9 — Upgrade success cohort
**Answer: 55 users · average 64.96 days from signup to first upgrade**

'Earliest' = `ROW_NUMBER()` by `start_date`; 'upgrade' = a later row on Premium/Family; 'still active' = a subscription open on 30-Jun-2024.

In [11]:
query = """
-- Q9: Jan-2024 signups who started Basic, upgraded, and are still active on 30-Jun-2024
WITH ranked AS (
    SELECT s.*,
           ROW_NUMBER() OVER (PARTITION BY s.user_id ORDER BY s.start_date, s.subscription_id) AS rn
    FROM subscriptions s
),
jan_basic_starters AS (
    SELECT u.user_id, u.signup_date
    FROM users u
    JOIN ranked r ON r.user_id = u.user_id AND r.rn = 1
    WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-01-31'
      AND r.plan = 'Basic'
),
first_upgrade AS (
    SELECT b.user_id, b.signup_date, MIN(r.start_date) AS first_upgrade_date
    FROM jan_basic_starters b
    JOIN ranked r ON r.user_id = b.user_id
                 AND r.rn > 1
                 AND r.plan IN ('Premium', 'Family')
    GROUP BY b.user_id, b.signup_date
)
SELECT COUNT(*)                                                    AS upgraded_users,
       ROUND(AVG(DATEDIFF(f.first_upgrade_date, f.signup_date)), 2) AS avg_days_signup_to_upgrade
FROM first_upgrade f
WHERE EXISTS (SELECT 1 FROM subscriptions s
              WHERE s.user_id = f.user_id
                AND s.status = 'active'
                AND s.start_date <= '2024-06-30'
                AND (s.end_date IS NULL OR s.end_date > '2024-06-30'));
"""
pd.read_sql(query, engine)

,upgraded_users,avg_days_signup_to_upgrade
0,55,64.96


## Q10 — Cliffhanger comebacks
**Answer: 4,345 cliffhanger comeback events**

**Top show: S088 — Rayalaseema Raga (64 comebacks)**

Each unique (user, show, incomplete_date) is counted once via `DISTINCT`.

In [12]:
query = """
-- Q10: Cliffhanger comebacks (incomplete session, same show again within 1-7 days)
WITH events AS (
    SELECT DISTINCT a.user_id, a.show_id, a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
      ON  b.user_id = a.user_id
      AND b.show_id = a.show_id
      AND b.session_date BETWEEN DATE_ADD(a.session_date, INTERVAL 1 DAY)
                             AND DATE_ADD(a.session_date, INTERVAL 7 DAY)
    WHERE a.completed = 0
)
SELECT (SELECT COUNT(*) FROM events) AS total_events,
       e.show_id,
       sh.title,
       COUNT(*)                      AS comebacks
FROM events e
JOIN shows sh ON sh.show_id = e.show_id
GROUP BY e.show_id, sh.title
ORDER BY comebacks DESC, e.show_id
LIMIT 5;
"""
pd.read_sql(query, engine)

,total_events,show_id,title,comebacks
0,4345,S088,Rayalaseema Raga,64
1,4345,S005,Show Special 27,61
2,4345,S019,Founder Diaries,61
3,4345,S024,Show Special 18,59
4,4345,S065,Show Special 21,58


## Q11 — Consecutive-week engagement
**Answer: 1,675 users with a 4+ week streak**

**Longest streak: 26 weeks** (every week of the period), e.g. user **U00213** (5 shown; more users tie).

Gaps-and-islands: distinct (user, week) rows; `week_no − ROW_NUMBER()` is constant within a consecutive run. Weeks are Monday-based (ISO) and ordinal from 2024-01-01 (a Monday), so year boundaries can't break streaks.

In [13]:
query = """
-- Q11: 4+ consecutive ISO weeks of activity (gaps-and-islands)
WITH user_weeks AS (             -- one row per user per active week; week_no is a Monday-based ordinal
    SELECT DISTINCT user_id,
           FLOOR(DATEDIFF(DATE_SUB(session_date, INTERVAL WEEKDAY(session_date) DAY), '2024-01-01') / 7) AS week_no
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered AS (
    SELECT user_id, week_no,
           week_no - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY week_no) AS island_key
    FROM user_weeks
),
streaks AS (
    SELECT user_id, island_key, COUNT(*) AS streak_weeks
    FROM numbered
    GROUP BY user_id, island_key
),
best AS (
    SELECT user_id, MAX(streak_weeks) AS longest_streak
    FROM streaks
    GROUP BY user_id
)
SELECT (SELECT COUNT(*) FROM best WHERE longest_streak >= 4)  AS users_with_4plus_streak,
       (SELECT MAX(longest_streak) FROM best)                 AS longest_streak_weeks,
       user_id                                                AS example_user
FROM best
WHERE longest_streak = (SELECT MAX(longest_streak) FROM best)
ORDER BY user_id
LIMIT 5;
"""
pd.read_sql(query, engine)

,users_with_4plus_streak,longest_streak_weeks,example_user
0,1675,26,U00213
1,1675,26,U00529
2,1675,26,U01259
3,1675,26,U01658
4,1675,26,U01793


## Q12 — Churn signal detection
**Answer: 521 churn-signal users**

June minutes dropped ≥50% vs May (only users with May activity). 97 of them watched nothing at all in June (100% drop). Window/aggregate is wrapped in CTEs and filtered in the outer query.

In [14]:
query = """
-- Q12: Churn signals - June watch minutes down 50 percent or more vs May
WITH monthly AS (
    SELECT user_id,
           SUM(CASE WHEN session_date BETWEEN '2024-05-01' AND '2024-05-31' THEN watch_minutes ELSE 0 END) AS may_minutes,
           SUM(CASE WHEN session_date BETWEEN '2024-06-01' AND '2024-06-30' THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
    GROUP BY user_id
),
with_drop AS (
    SELECT user_id, may_minutes, june_minutes,
           ROUND(100 * (may_minutes - june_minutes) / may_minutes, 2) AS drop_pct
    FROM monthly
    WHERE may_minutes > 0
)
SELECT w.user_id, u.name, w.may_minutes, w.june_minutes, w.drop_pct
FROM with_drop w
JOIN users u ON u.user_id = w.user_id
WHERE w.drop_pct >= 50
ORDER BY w.drop_pct DESC, w.user_id;
"""
pd.read_sql(query, engine)


,user_id,name,may_minutes,june_minutes,drop_pct
0,U00023,Amit Mukherjee,43.0,0.0,100.00
1,U00166,Shaurya Malhotra,94.0,0.0,100.00
2,U00211,Ravi Menon,336.0,0.0,100.00
3,U00225,Kritika Patil,209.0,0.0,100.00
4,U00237,Shaurya Bansal,126.0,0.0,100.00
...,...,...,...,...,...
516,U01858,Sanjay Mukherjee,296.0,147.0,50.34
517,U02530,Nandini Roy,451.0,224.0,50.33
518,U01192,Rohit Mukherjee,136.0,68.0,50.00
519,U01806,Yuvraj Raman,78.0,39.0,50.00


In [15]:
# Total churn-signal users
churn = pd.read_sql(query, engine)
print("Total churn-signal users:", len(churn))

Total churn-signal users: 521
